# 学术文献 RAG 系统进阶优化

> 从 Naive RAG 到 Advanced RAG 的全链路升级实操

本 Notebook 覆盖：
- 数据端优化：四层深度清洗、元数据丰富、文本切片策略
- 检索端准备：arXiv 批量数据获取、向量库入库
- 评估体系：测试集自动构建

基于百度千帆平台 (ERNIE + bge-large-en Embedding)

---
## 0. 环境准备

In [ ]:
# 依赖安装（按需执行）
!pip install langchain langchain-community langchain-text-splitters langchain-openai langchain-pymupdf4llm
!pip install sentence-transformers chromadb python-dotenv
!pip install arxiv aiohttp aiofiles pandas numpy
!pip install PyMuPDF

In [ ]:
# 基础导入
import os
import re
import json
import hashlib
import time
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    MarkdownTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import chromadb

load_dotenv()
api_key = os.getenv('API_KEY')

In [ ]:
# 配置 LLM（百度千帆 ERNIE）
llm = ChatOpenAI(
    api_key=api_key,
    model_name="ernie-5.0",
    base_url="https://qianfan.baidubce.com/v2"
)

In [ ]:
# 配置 Embedding（bge-large-en）
class Embedding:
    def __init__(self):
        self.client = OpenAI(
            api_key=api_key,
            base_url="https://qianfan.baidubce.com/v2"
        )
        self.BATCH_SIZE = 16

    def embed_documents(self, documents: list):
        """文档向量化"""
        truncated = [d for d in documents]
        all_embeddings = []
        for i in range(0, len(truncated), self.BATCH_SIZE):
            batch = truncated[i:i + self.BATCH_SIZE]
            response = self.client.embeddings.create(
                model='bge-large-en',
                input=batch
            )
            all_embeddings.extend([item.embedding for item in response.data])
        return all_embeddings

    def embed_query(self, query: str):
        """查询向量化（LangChain 检索时必须实现此方法）"""
        return self.embed_documents([query])[0]

embeddings = Embedding()

---
## 1. arXiv 批量获取论文数据

### 1.1 arxiv.py 基础用法

In [ ]:
import arxiv

# 客户端初始化
client = arxiv.Client()

# 自定义客户端（大规模采集时使用）
custom_client = arxiv.Client(
    page_size=100,        # 单次API请求获取的最大结果数，默认100，上限2000
    delay_seconds=3.0,    # 两次API请求之间的等待间隔（秒），遵守arXiv使用条款
    num_retries=3         # 请求失败时的重试次数
)

In [ ]:
# 构建查询 — 按关键词
search = arxiv.Search(
    query="attention",
    max_results=10,
    sort_by=arxiv.SortCriterion.SubmittedDate,   # 按提交日期排序
    sort_order=arxiv.SortOrder.Descending        # 降序排列（最新优先）
)

# 按字段限定查询：作者 + 分类 + 标题
search = arxiv.Search(
    query="au:Kaiming_He AND cat:cs.CV AND ti:ResNet",
    max_results=20
)

# 按论文 ID 精确定位
search_by_id = arxiv.Search(id_list=["2301.07041", "2106.09685"])

In [ ]:
# 获取结果（返回生成器，逐条产出，避免内存溢出）
results = client.results(search)

for paper in results:
    print(f"标题: {paper.title}")
    print(f"作者: {', '.join(a.name for a in paper.authors)}")
    print(f"摘要: {paper.summary}")
    print(f"提交日期: {paper.published}")
    print(f"PDF链接: {paper.pdf_url}")
    print("---")

In [ ]:
# 下载单篇 PDF
paper = next(client.results(search))
paper.download_pdf(dirpath="./pdfs", filename="sample_paper.pdf")

### 1.2 异步批量下载

In [ ]:
import asyncio
import aiohttp
import aiofiles
import logging
from dataclasses import dataclass

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ==================== 可配置参数 ====================
SAVE_DIR = "./nlp_papers"
MAX_RESULTS = 200
MAX_CONCURRENT = 5
PAGE_SIZE = 100
DELAY_SECONDS = 3.0
NUM_RETRIES = 3
SORT_BY = "Relevance"
QUERY = "cat:cs.CL"
# ===================================================

@dataclass
class PaperInfo:
    entry_id: str
    title: str
    pdf_url: str

def sanitize_filename(title: str) -> str:
    """将论文标题转换为安全文件名"""
    safe = re.sub(r'[^\w\s\-]', '', title)
    safe = safe.strip()[:100]
    return safe

async def download_paper(session, paper: PaperInfo, save_dir: Path, semaphore: asyncio.Semaphore):
    """异步下载单篇论文"""
    filename = sanitize_filename(paper.title) + ".pdf"
    pdf_path = save_dir / filename

    if pdf_path.exists():
        logging.info(f"跳过 (已存在): {paper.title}")
        return True

    async with semaphore:
        for attempt in range(NUM_RETRIES):
            try:
                async with session.get(paper.pdf_url) as response:
                    if response.status == 200:
                        async with aiofiles.open(pdf_path, 'wb') as f:
                            await f.write(await response.read())
                        logging.info(f"下载成功: {paper.title}")
                        return True
                    else:
                        logging.warning(f"HTTP {response.status}: {paper.title}, 重试 {attempt+1}/{NUM_RETRIES}")
            except Exception as e:
                logging.error(f"异常: {paper.title} - {e}, 重试 {attempt+1}/{NUM_RETRIES}")
            await asyncio.sleep(2 ** attempt)
    logging.error(f"彻底失败: {paper.title}")
    return False

async def main():
    save_dir = Path(SAVE_DIR)
    save_dir.mkdir(exist_ok=True)

    sort_map = {
        "Relevance": arxiv.SortCriterion.Relevance,
        "SubmittedDate": arxiv.SortCriterion.SubmittedDate,
        "LastUpdatedDate": arxiv.SortCriterion.LastUpdatedDate,
    }
    sort_criterion = sort_map.get(SORT_BY, arxiv.SortCriterion.Relevance)

    client = arxiv.Client(
        page_size=PAGE_SIZE,
        delay_seconds=DELAY_SECONDS,
        num_retries=NUM_RETRIES
    )

    search = arxiv.Search(
        query=QUERY,
        max_results=MAX_RESULTS,
        sort_by=sort_criterion
    )

    logging.info("开始获取论文元数据...")
    papers_info = []
    for paper in client.results(search):
        papers_info.append(PaperInfo(
            entry_id=paper.entry_id,
            title=paper.title,
            pdf_url=paper.pdf_url
        ))
    logging.info(f"获取到 {len(papers_info)} 篇论文")

    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    async with aiohttp.ClientSession() as session:
        tasks = [download_paper(session, paper, save_dir, semaphore) for paper in papers_info]
        results = await asyncio.gather(*tasks)

    success = sum(results)
    logging.info(f"下载完成: 成功 {success} 篇, 失败 {len(papers_info)-success} 篇")

# 运行（在 Jupyter 中需要特殊处理异步）
# await main()

---
## 2. 正则表达式 — 文本清洗核心工具

### 2.1 四个基础函数

In [ ]:
import re

# --- re.search: 在字符串中查找第一个匹配 ---
text = "小坤今年18岁，他喜欢唱跳RAP打篮球，最喜欢的是打篮球。"
result = re.search(r'篮球', text)
if result:
    print(result.group())       # 匹配到的文字
    print(result.span())        # 匹配位置

In [ ]:
# --- re.match vs re.search ---
print(re.match(r'小坤', text).group())   # 成功，从开头匹配
print(re.match(r'18', text))              # None，不是从开头
print(re.search(r'18', text).group())     # 成功，可以从中间开始

In [ ]:
# --- re.findall: 找到所有非重叠匹配（最实用）---
text2 = "苹果 香蕉 苹果 橙子 苹果"
print(re.findall(r'苹果', text2))  # ['苹果', '苹果', '苹果']

text3 = "小明18岁，小红20岁，小刚19岁"
ages = re.findall(r'\d+', text3)    # \d+ = 一个或多个数字
print(ages)                           # ['18', '20', '19']

In [ ]:
# --- re.sub: 替换/删除 ---
text4 = "我喜欢吃苹果和香蕉。"
print(re.sub(r'苹果', '芒果', text4))  # 我喜欢吃芒果和香蕉。

text5 = "价格是：￥128元"
print(re.sub(r'￥', '', text5))         # 价格是：128元

### 2.2 字符类与预定义字符类

In [ ]:
t = "小坤今年18岁，他喜欢唱跳RAP打篮球。"

print(re.search(r'[0-9]', t).group())      # 1  (匹配任意一个数字)
print(re.search(r'[a-zA-Z]', t).group())   # R  (匹配任意一个字母)
print(re.search(r'[^0-9]', t).group())     # 小 (匹配除数字以外的字符)

In [ ]:
# 预定义字符类速查
# \d = [0-9]          任意数字
# \D = [^0-9]         非数字
# \w = [a-zA-Z0-9_]   单词字符
# \W = [^a-zA-Z0-9_]  非单词字符
# \s = [ \t\n\r\f\v]  空白字符
# \S = [^ \t\n\r\f\v] 非空白字符
# \b                  单词边界

text6 = "小坤18岁，邮箱：kun_18@example.com"
print(re.findall(r'\d+', text6))   # ['18']
print(re.findall(r'\w+', text6))   # ['小坤18岁', '邮箱', 'kun_18', 'example', 'com']

### 2.3 量词

In [ ]:
# 量词速查
# *  →  0次或多次（≥0）
# +  →  1次或多次（≥1）
# ?  →  0次或1次
# {n}    →  恰好n次
# {n,}   →  至少n次
# {n,m}  →  n到m次

text7 = "小坤今年18岁，成绩89.5分，电话13800138000。"

print(re.findall(r'\d+', text7))         # ['18', '89', '5', '13800138000']
print(re.search(r'\d+\.?\d*', text7).group())  # 18 (小数可选)
print(re.search(r'\d{2,}', text7).group())       # 18 (至少2个连续数字)
print(re.search(r'\d{11}', text7).group())        # 13800138000 (11位电话号码)

### 2.4 捕获组与反向引用

In [ ]:
text8 = "小坤今年18岁，他喜欢唱跳RAP打篮球。"

# 捕获并提取
result = re.search(r'(\d+)岁', text8)
print(result.group(0))    # 整个匹配：18岁
print(result.group(1))    # 第1组：18

result2 = re.search(r'(\w+)喜欢(\w+)', text8)
print(result2.group(1))   # 他
print(result2.group(2))   # 唱跳RAP打篮球

In [ ]:
# 反向引用：修复断裂单词（最常见场景）
text9 = "atten- tion 以及     pro- cess"
cleaned = re.sub(r'(\w+)-\s*(\w+)', r'\1\2', text9)
print(cleaned)   # attention 以及 process

### 2.5 贪婪 vs 非贪婪匹配

In [ ]:
text10 = "开始 $x^2 + y^2$ 结束 $a + b$ 结束"

# 贪婪匹配（默认）：匹配到最后一个 $
result_greedy = re.search(r'\$.+\$', text10)
print("贪婪:", result_greedy.group())   # $x^2 + y^2$ 结束 $a + b$

# 非贪婪匹配（加 ?）：仅匹配到第一个 $
result_non_greedy = re.search(r'\$.+?\$', text10)
print("非贪婪:", result_non_greedy.group())   # $x^2 + y^2$

---
## 3. 四层深度清洗（针对 arXiv 双栏 PDF）

清洗架构：**基础清洗 → 页眉页脚清理 → 基础去重 → 语义去重**

In [ ]:
# ============ 第一层：基础清洗 ============
def basic_clean(text: str) -> str:
    """第一层：基础清洗（含断裂修复 + 公式修复）"""
    # 1. 统一编码与空白压缩
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    text = re.sub(r'\s+', ' ', text).strip()

    # 2. 修复文本断裂（双栏PDF最常见问题）
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)     # atten- tion → attention
    text = re.sub(r'(\w+)\s*-\s*(\w+)', r'\1\2', text)

    # 3. 修复公式（保留核心数学内容，去除多余LaTeX标记）
    text = re.sub(r'\$\s*(.+?)\s*\$', r'\1', text)           # 行内公式保留内容
    text = re.sub(r'\\\[(.+?)\\\]', r'\1', text, flags=re.DOTALL)   # 行间公式
    text = re.sub(r'\\[a-zA-Z]+\s*', '', text)               # 清理多余LaTeX命令

    # 4. 字符级标准化
    text = re.sub(r'(.)\1{4,}', r'\1\1\1', text)             # 连续重复字符压缩
    return text

In [ ]:
# ============ 第二层：页眉、页脚、作者、残留清理 ============
def clean_header_footer_author(text: str) -> str:
    """第二层：页眉、页脚、作者、时间戳、ID编码、乱码全面清理"""
    # 1. arXiv ID编码 + 版本号 + 类别编码
    text = re.sub(r'arXiv:\s*\d+\.\d+[v\d]*\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\s*cs\.[A-Z]+\s*\]', '', text, flags=re.IGNORECASE)

    # 2. 页脚常见模式
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)              # 单独页码行
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*-\s*\d+\s*-\s*$', '', text, flags=re.MULTILINE)    # - 12 - 型页脚

    # 3. 时间戳（Submitted / Received / Published 等）
    text = re.sub(r'(Submitted|Received|Published|Accepted|Revised)\s+on\s+.*?(?=\n\n|\Z)',
                  '', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'\d{1,2}\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+\d{4}',
                  '', text, flags=re.IGNORECASE)
    text = re.sub(r'\d{4}-\d{2}-\d{2}', '', text)                            # 2025-03-01 型日期

    # 4. 作者、邮箱、机构、DOI、版权等残留
    text = re.sub(r'(?i)(author|authors|corresponding author|affiliation).*?(?=\n\n|\Z)',
                  '', text, flags=re.DOTALL)
    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '', text)                     # 邮箱
    text = re.sub(r'DOI:\s*\S+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'©\s*\d+|Copyright.*|All rights reserved.*', '', text, flags=re.IGNORECASE)

    # 5. 图表引用行
    text = re.sub(r'(?i)^\s*(Figure|Table|Algorithm|Eq\.)\s*\d+[:.]?\s*.*?(?=\n\n|\Z)',
                  '', text, flags=re.MULTILINE | re.DOTALL)

    # 6. 乱码与非ASCII字符
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)                               # 清除特殊符号
    text = re.sub(r'[\u2000-\u200F\u2028\u2029]', ' ', text)                # 特殊Unicode控制字符

    return text.strip()

In [ ]:
# ============ 第三层：基础去重 ============
def basic_deduplication(text: str) -> str:
    """第三层：基础去重（字符级 + 段落级）"""
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 30]
    seen = {}
    cleaned = []

    for para in paragraphs:
        # 特征提取：MD5哈希 + 前150字符
        key = hashlib.md5(para[:150].encode('utf-8')).hexdigest()
        if key not in seen:
            seen[key] = para
            cleaned.append(para)

    return '\n\n'.join(cleaned)

In [ ]:
# ============ 第四层：语义去重（使用 bge-large-en）============
def semantic_deduplication(text: str, embeddings_obj, similarity_threshold: float = 0.88) -> str:
    """第四层：语义去重（使用bge-large-en）"""
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 50]
    if not paragraphs:
        return ""

    # 计算每个段落的Embedding
    embeddings_list = embeddings_obj.embed_documents(paragraphs)

    unique_paras = []
    unique_embs = []

    for para, emb in zip(paragraphs, embeddings_list):
        is_duplicate = False
        for unique_emb in unique_embs:
            # 计算余弦相似度
            similarity = sum(a * b for a, b in zip(emb, unique_emb)) / (
                (sum(x * x for x in emb) ** 0.5) * (sum(y * y for y in unique_emb) ** 0.5) + 1e-8
            )
            if similarity > similarity_threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            unique_paras.append(para)
            unique_embs.append(emb)

    return '\n\n'.join(unique_paras)

In [ ]:
# ============ 完整主函数（四层串联）============
def clean_double_column_pdf(raw_text: str, embeddings_obj=None) -> str:
    """针对arXiv双栏PDF的完整四层深度清洗主函数"""
    # 第1层：基础清洗（断裂 + 公式）
    text = basic_clean(raw_text)

    # 第2层：页眉页脚作者残留清理
    text = clean_header_footer_author(text)

    # 第3层：基础去重
    text = basic_deduplication(text)

    # 第4层：语义去重（需传入embedding对象）
    if embeddings_obj:
        text = semantic_deduplication(text, embeddings_obj)

    # 最终段落规范化
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

---
## 4. 元数据获取与注入

### 4.1 批量获取 arXiv 论文元数据

In [ ]:
import fitz  # PyMuPDF

# ================= 参数 =================
PDF_DIR = Path("./nlp_papers")
BATCH_SIZE = 5          # 批量查询
DELAY = 5               # 防429限流
# ======================================

def extract_arxiv_id_from_filename(filename):
    match = re.search(r'(\d{4}\.\d{4,5})', filename)
    return match.group(1) if match else None

def extract_arxiv_id_from_pdf(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        text = doc[0].get_text()
        match = re.search(r'arXiv:\s*(\d{4}\.\d{4,5})', text)
        return match.group(1) if match else None
    except:
        return None

# ================= Step 1: 收集待处理 =================
pdf_files = list(PDF_DIR.glob("*.pdf"))
print(f"找到 {len(pdf_files)} 个 PDF")
id_to_pdf = {}
todo_ids = []

for pdf_path in pdf_files:
    json_path = pdf_path.with_suffix(".json")
    if json_path.exists():
        continue
    arxiv_id = extract_arxiv_id_from_filename(pdf_path.name)
    if not arxiv_id:
        arxiv_id = extract_arxiv_id_from_pdf(pdf_path)
    if not arxiv_id:
        print(f"无法提取 ID: {pdf_path.name}")
        continue
    id_to_pdf[arxiv_id] = pdf_path
    todo_ids.append(arxiv_id)

todo_ids = list(set(todo_ids))
print(f"待处理数量: {len(todo_ids)}")

# ================= Step 2: 批量查询 =================
client = arxiv.Client(page_size=10, delay_seconds=3, num_retries=2)
success = 0

for i in range(0, len(todo_ids), BATCH_SIZE):
    batch_ids = todo_ids[i:i + BATCH_SIZE]
    print(f"\n查询 batch: {batch_ids}")
    search = arxiv.Search(id_list=batch_ids)
    try:
        results = list(client.results(search))
    except Exception as e:
        print("查询失败:", e)
        continue

    for paper in results:
        raw_id = paper.get_short_id()
        arxiv_id = raw_id.split("v")[0]
        if arxiv_id not in id_to_pdf:
            continue
        pdf_path = id_to_pdf[arxiv_id]
        json_path = pdf_path.with_suffix(".json")

        data = {
            "arxiv_id": arxiv_id,
            "title": paper.title,
            "authors": [a.name for a in paper.authors],
            "published": str(paper.published),
            "updated": str(paper.updated),
            "summary": paper.summary,
            "categories": paper.categories,
            "primary_category": paper.primary_category,
        }

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"✔ 已保存: {arxiv_id}")
        success += 1

    time.sleep(DELAY)

print(f"\n完成 {success} 条")

### 4.2 元数据注入到 Chunk

In [ ]:
def load_metadata(json_path: Path) -> dict:
    """读取同名json文件中的元数据"""
    if not json_path.exists():
        return {}
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def merge_metadata_and_tag(chunks: list, json_metadata: dict) -> list:
    """metadata合并 + 评估标签注入"""
    tagged_docs = []
    for i, chunk in enumerate(chunks):
        arxiv_id = json_metadata.get("arxiv_id", "unknown")
        chunk_id = f"{arxiv_id}_chunk_{i:03d}"          # 全局唯一，永不改变

        merged_meta = {
            "chunk_id": chunk_id,
            "arxiv_id": json_metadata.get("arxiv_id"),
            "title": json_metadata.get("title"),
            "authors": json_metadata.get("authors", []),
            "published": json_metadata.get("published"),
            "updated": json_metadata.get("updated"),
            "categories": json_metadata.get("categories", []),
            "source": chunk.metadata.get("source"),
        }

        chunk.metadata.update(merged_meta)
        tagged_docs.append(chunk)
    return tagged_docs

---
## 5. 文本切片策略

### 5.1 Markdown 结构化切片（技术文档）

In [ ]:
# 基本用法
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "一级标题"),
        ("##", "二级标题"),
        ("###", "三级标题"),
    ],
    strip_headers=False
)

markdown_document = """
# Foo
## Bar
Hi this is Jim
Hi this is Joe
## Baz
Hi this is Molly
"""

splits = markdown_splitter.split_text(markdown_document)
for split in splits:
    print(f"元数据: {split.metadata}")
    print(f"内容: {split.page_content}\n")

In [ ]:
# 完整版：技术文档专用切片器
def split_markdown_document(md_text: str, chunk_size: int = 512, overlap: int = 50):
    """技术文档专用切片器"""
    # 1. 按 H2/H3 标题切分
    headers_to_split_on = [
        ("##", "章节"),
        ("###", "小节"),
    ]
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
    header_splits = md_splitter.split_text(md_text)

    # 2. 递归切分过长章节
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", "。", "；", " ", ""],
    )
    final_chunks = text_splitter.split_documents(header_splits)
    return final_chunks

# 使用示例
# with open('家用电视技术说明文档.md', 'r', encoding='utf-8') as f:
#     docs = f.read()
# chunks = split_markdown_document(docs)
# chunk_lengths = [len(chunk.page_content) for chunk in chunks]
# print(f"切片数量: {len(chunks)}")
# print(f"最小长度: {min(chunk_lengths)} 字符")
# print(f"最大长度: {max(chunk_lengths)} 字符")
# print(f"平均长度: {np.mean(chunk_lengths):.1f} 字符")

### 5.2 法律条文切片（基于规则）

In [ ]:
class LawTextSplitter:
    """法律条文专用切片器（基于规则）"""

    def __init__(self, max_characters: int = 614, overlap_characters: int = 0):
        self.max_chars = max_characters
        self.overlap_chars = overlap_characters

        # 条款编号正则（支持中文数字、阿拉伯数字、项/款）
        self.pattern = re.compile(
            r'(第[零一二三四五六七八九十百千万\d]+[条款项]|'
            r'第[\(（]\d+[\)）]项|'
            r'Article\s+\d+)'
        )

    def split_text(self, text: str) -> List[Dict[str, Any]]:
        """按条款编号切分法律文本"""
        parts = []
        last_end = 0

        for match in self.pattern.finditer(text):
            start = match.start()
            end = match.end()

            # 1. 收集编号之前的文本（前言、目录等）
            if start > last_end:
                preamble = text[last_end:start].strip()
                if preamble:
                    parts.append({"clause_num": "", "content": preamble})

            # 2. 记录当前条款编号
            clause_num = match.group(0)
            parts.append({"clause_num": clause_num, "content": ""})
            last_end = end

        # 3. 处理文件末尾剩余文本
        if last_end < len(text):
            remaining = text[last_end:].strip()
            if remaining:
                parts.append({"clause_num": "", "content": remaining})

        # 4. 合并编号与内容
        chunks = []
        i = 0
        while i < len(parts):
            if parts[i]["clause_num"]:
                num = parts[i]["clause_num"]
                content_parts = []
                j = i + 1
                while j < len(parts) and not parts[j]["clause_num"]:
                    content_parts.append(parts[j]["content"])
                    j += 1
                full_content = num + " " + " ".join(content_parts).strip()
                i = j
            else:
                full_content = parts[i]["content"]
                i += 1

            # 5. 若超长则二次分割
            if len(full_content) > self.max_chars:
                sub_chunks = self._split_long_clause(full_content)
                chunks.extend(sub_chunks)
            else:
                chunks.append({
                    "text": full_content.strip(),
                    "metadata": {"clause_num": num if 'num' in locals() else ""}
                })
        return chunks

    def _split_long_clause(self, text: str) -> List[Dict[str, Any]]:
        """对超长条款按句子分割"""
        first_line = text.split('\n')[0]
        clause_num = ""
        if self.pattern.match(first_line):
            clause_num = self.pattern.match(first_line).group(0)

        sentences = re.split(r'(?<=[。；！？])\s*', text)
        chunks = []
        current = ""
        for sent in sentences:
            if len(current) + len(sent) <= self.max_chars:
                current += sent
            else:
                if current:
                    chunks.append({
                        "text": current.strip(),
                        "metadata": {"clause_num": clause_num}
                    })
                current = sent
        if current:
            chunks.append({
                "text": current.strip(),
                "metadata": {"clause_num": clause_num}
            })
        return chunks


# 使用示例
# with open('./中华人民共和国个人信息保护法_20210820.txt', 'r', encoding='UTF-8') as f:
#     doc = f.read()
# splitter = LawTextSplitter(max_characters=614)
# chunks = splitter.split_text(doc)
# for i, chunk in enumerate(chunks[:5]):
#     print(f"Chunk {i + 1}: {chunk['metadata']}")
#     print(chunk['text'])
#     print("---")

### 5.3 结构化数据切片（CSV/表格）

In [ ]:
def csv_to_chunks(file_path: str) -> list:
    df = pd.read_csv(file_path)
    chunks = []
    for idx, row in df.iterrows():
        # 将一行转为 "列名: 值；列名: 值" 格式
        row_text = "；".join([f"{col}: {row[col]}" for col in df.columns if pd.notna(row[col])])
        chunks.append({"text": row_text, "metadata": {"row_index": idx}})
    return chunks

# 使用示例
# chunks = csv_to_chunks('百度模型.csv')
# chunk_lengths = [len(chunk['text']) for chunk in chunks]
# print(f"切片数量: {len(chunks)}")
# print(f"平均长度: {np.mean(chunk_lengths):.1f} 字符")

### 5.4 本项目方案：Markdown 感知 + 标题层级切片

使用 PyMuPDF4LLM 将 PDF 转为 Markdown 后，结合 Markdown 结构和递归切片：

In [ ]:
def markdown_aware_chunking(raw_text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> list:
    """推荐方案：Markdown感知 + 递归切片（保留标题层级）"""

    # 1. 先用MarkdownTextSplitter按标题层级切分（保留结构）
    markdown_splitter = MarkdownTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    md_chunks = markdown_splitter.split_text(raw_text)

    # 2. 对每个Chunk提取标题层级作为metadata
    final_chunks = []
    current_section = "Introduction"   # 默认标题

    for chunk in md_chunks:
        # 提取当前Chunk的标题（# ## ### 等）
        title_match = re.search(r'^(#{1,6})\s+(.+)$', chunk, re.MULTILINE)
        if title_match:
            current_section = title_match.group(2).strip()

        doc = Document(
            page_content=chunk,
            metadata={
                "section_title": current_section,     # 关键！标题层级信息
                "section_level": len(title_match.group(1)) if title_match else 0,
            }
        )
        final_chunks.append(doc)

    # 3. 可选：再进行一次小粒度递归切片（防止超长Chunk）
    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    final_docs = []
    for doc in final_chunks:
        sub_chunks = recursive_splitter.split_text(doc.page_content)
        for sub in sub_chunks:
            final_docs.append(Document(
                page_content=sub,
                metadata=doc.metadata.copy()   # 继承标题信息
            ))

    return final_docs

---
## 6. 数据入库（ChromaDB）

完整的入库流程：加载 PDF → 深度清洗 → Markdown 感知切片 → 合并元数据 → 向量化入库

In [ ]:
# ================== 1. 配置 ==================
paper_dir = Path("./pdf_data")
db_dir = "./final_db"
collection_name = "nlp_papers"
batch_size = 20
scan_path = './scan.json'

# ================== 2. 初始化 Chroma ==================
client = chromadb.PersistentClient(path=db_dir)

db = Chroma(
    client=client,
    collection_name=collection_name,
    embedding_function=embeddings
)

In [ ]:
# ================== 3. 入库文件扫描 ==================
folder = Path('./pdf_data')
json_stems = {p.stem for p in folder.glob("*.json")}
with_json = set()
without_json = set()
for pdf in folder.glob("*.pdf"):
    (with_json if pdf.stem in json_stems else without_json).add(pdf.stem)
result = {"1": sorted(with_json), "0": sorted(without_json), "2": []}

print(f'一共{len(result["0"])}篇论文没有json配对文件')
print(f'一共{len(result["1"])}篇论文有json配对文件')

with open('./scan.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False)

In [ ]:
# ================== 4. 正式入库 ==================
with open(scan_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
pdf_list = data['1']

success_count = 0
for name in pdf_list[:]:
    pdf_name = paper_dir / Path(name)
    pdf_path = pdf_name.with_suffix('.pdf')
    json_path = pdf_name.with_suffix('.json')

    # 3.1 读取 + 清洗 + 切片
    loader = PyMuPDF4LLMLoader(pdf_path)
    doc = loader.load()
    raw_text = "\n".join([p.page_content for p in doc])
    cleaned_text = clean_double_column_pdf(raw_text)
    chunks = markdown_aware_chunking(cleaned_text)
    if not chunks:
        continue

    # 3.2 读取metadata
    with open(json_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    # 3.3 metadata数据融合 + 评估标签注入
    tagged_chunks = merge_metadata_and_tag(chunks, json_data)

    # 3.4 向量化入库
    before = len(db.get()['ids'])
    for i in range(0, len(tagged_chunks), batch_size):
        batch = tagged_chunks[i:i+batch_size]
        db.add_documents(batch)
    after = len(db.get()['ids'])
    added = after - before
    print(f'{pdf_name} 成功写入{added}个Chunk (累计{after}个)')

    if added:
        name = data['1'].pop(0)
        data['2'].append(name)
        success_count += 1

with open(scan_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False)

print("全部论文入库完成！")
print(f"成功处理论文数: {success_count} / {len(pdf_list)}")
print(f"还未处理论文数：{len(data['1'])}")

---
## 7. 测试集自动构建

解决 RAG 评估冷启动问题：自动生成 QA 对 → 质量过滤 → 保存为标准化测试集。

每个样本包含：`query` + `answer` + `gold_chunks` + `difficulty`

In [ ]:
import random

def build_testset(db, llm, num_samples=250, questions_per_chunk=4):
    """从已入库的向量库中自动构建测试集"""
    # 1. 随机抽取Chunk
    all_docs = db.get(limit=0)["documents"]
    sampled_docs = random.sample(all_docs, min(num_samples, len(all_docs)))

    testset = []

    for doc in sampled_docs:
        chunk_text = doc.page_content
        chunk_id = doc.metadata.get("chunk_id")

        # 2. LLM生成问题、答案、gold_chunks和难度
        prompt = f"""
基于以下论文片段，生成{questions_per_chunk}个高质量问题。

要求：
- 问题必须可以从文本中完全回答
- 避免过于简单或直接复制原文
- 覆盖定义、方法、实验、对比等不同类型
- 每个问题输出对应的1~3个gold chunk_id（当前chunk必须包含在内）
- 标注难度：easy / medium / hard

文本：
{chunk_text}

请严格按以下JSON格式输出：
{{
    "questions": [
        {{
            "query": "问题1",
            "answer": "标准答案1",
            "gold_chunks": ["{chunk_id}", "其他相关chunk_id"],
            "difficulty": "medium"
        }}
    ]
}}
"""
        response = llm.invoke(prompt)
        try:
            data = json.loads(response)
            for q in data.get("questions", []):
                testset.append({
                    "query_id": f"q{len(testset)+1:03d}",
                    "query": q["query"],
                    "answer": q["answer"],
                    "gold_chunks": q["gold_chunks"],
                    "type": "auto_generated",
                    "difficulty": q.get("difficulty", "medium")
                })
        except:
            continue

    # 保存测试集
    with open("eval_testset.json", "w", encoding="utf-8") as f:
        json.dump(testset, f, ensure_ascii=False, indent=2)

    print(f"测试集构建完成！共生成 {len(testset)} 条测试数据")
    return testset


# 小规模测试
# testset = build_testset(db, llm, num_samples=5, questions_per_chunk=2)

---
## 8. 优化效果总结

| 维度 | Naive RAG | Advanced RAG（本实验） |
|------|----------|----------------------|
| **数据量** | 10~20篇 | 200篇（可扩展至数万） |
| **数据清洗** | 基础正则（断词修复） | 四层深度清洗（断裂修复+公式+页眉页脚+去重） |
| **元数据** | source + chunk_index | 标题、作者、arXiv ID、日期、类别等完整信息 |
| **切片策略** | 纯递归切片 (1500字符) | Markdown感知 + 标题层级 + 递归组合切片 |
| **评估体系** | 人工主观判断 | 自动化测试集 + MRR@K / Recall@K / NDCG@K |
| **可溯源** | ❌ | ✅ 完整出处标注 |

> 数据端优化到此结束。后续进入检索端优化（混合检索、查询改写、重排序）。